testing the datasets API

In [2]:
import requests
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

url = "https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml"
xml_bytes = requests.get(url, timeout=60).content
root = ET.fromstring(xml_bytes)

def localname(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

all_tags = [localname(elem.tag) for elem in root.iter()]
tag_counts = Counter(all_tags)

for tag, count in tag_counts.most_common(40):
    print(f"{tag}: {count}")

values: 108216
value: 108216
name: 59915
type: 49012
addressLine: 48300
text: 48300
connector: 42741
connectorType: 42741
chargingMode: 42741
connectorFormat: 42741
maxPowerAtSocket: 42741
refillPoint: 35765
maximumCurrent: 34527
voltage: 34400
authenticationAndIdentificationMethods: 25162
energyInfrastructureSite: 12075
lastUpdated: 12075
accessibility: 12075
operatingHours: 12075
label: 12075
overallPeriod: 12075
overallStartTime: 12075
locationReference: 12075
_locationReferenceExtension: 12075
facilityLocation: 12075
address: 12075
postcode: 12075
coordinatesForDisplay: 12075
latitude: 12075
longitude: 12075
operator: 12075
organisationUnit: 12075
energyInfrastructureStation: 12075
typeOfSite: 10864
supplementalFacility: 1442
serviceFacilityType: 1442
associatedFacility: 712
payload: 1
feedDescription: 1
publicationTime: 1


In [3]:
sites = [x for x in root.iter() if localname(x.tag) == "energyInfrastructureSite"]
print("Sites found:", len(sites))

first_site = sites[0]

def print_tree(elem, level=0, max_depth=4):
    if level > max_depth:
        return
    indent = "  " * level
    text = (elem.text or "").strip()
    text_preview = f" -> {text[:50]}" if text else ""
    print(f"{indent}{localname(elem.tag)}{text_preview}")
    for child in list(elem):
        print_tree(child, level + 1, max_depth)

print_tree(first_site, max_depth=5)

Sites found: 12075
energyInfrastructureSite
  name
    values
      value -> QWELLO - Calle Juan Antonio Zenón 90
  lastUpdated -> 2026-04-13T00:33:58.000+02:00
  accessibility
  operatingHours
    label
    overallPeriod
      overallStartTime -> 2023-01-01T00:00:00.000+01:00
  locationReference
    _locationReferenceExtension
      facilityLocation
        address
          postcode -> 28600
          addressLine
          addressLine
          addressLine
          addressLine
    coordinatesForDisplay
      latitude -> 40.28838
      longitude -> -4.020607
  operator
    name
      values
        value -> QWELLO España SL
    organisationUnit
  associatedFacility
    type -> parkingSite
  typeOfSite -> openSpace
  energyInfrastructureStation
    authenticationAndIdentificationMethods -> debitCard
    authenticationAndIdentificationMethods -> rfid
    authenticationAndIdentificationMethods -> nfc
    authenticationAndIdentificationMethods -> creditCard
    refillPoint
      name
   

In [4]:
def child_tag_counts(elem):
    return Counter(localname(child.tag) for child in list(elem))

print("Direct children of first site:")
print(child_tag_counts(first_site))

stations = [x for x in first_site.iter() if localname(x.tag) == "energyInfrastructureStation"]
refill_points = [x for x in first_site.iter() if localname(x.tag) == "refillPoint"]
connectors = [x for x in first_site.iter() if localname(x.tag) == "connector"]

print("stations in first site:", len(stations))
print("refill_points in first site:", len(refill_points))
print("connectors in first site:", len(connectors))

if stations:
    print("\nDirect children of first station:")
    print(child_tag_counts(stations[0]))

if refill_points:
    print("\nDirect children of first refillPoint:")
    print(child_tag_counts(refill_points[0]))

if connectors:
    print("\nDirect children of first connector:")
    print(child_tag_counts(connectors[0]))

Direct children of first site:
Counter({'name': 1, 'lastUpdated': 1, 'accessibility': 1, 'operatingHours': 1, 'locationReference': 1, 'operator': 1, 'associatedFacility': 1, 'typeOfSite': 1, 'energyInfrastructureStation': 1})
stations in first site: 1
refill_points in first site: 2
connectors in first site: 2

Direct children of first station:
Counter({'authenticationAndIdentificationMethods': 4, 'refillPoint': 2})

Direct children of first refillPoint:
Counter({'name': 1, 'connector': 1})

Direct children of first connector:
Counter({'connectorType': 1, 'chargingMode': 1, 'connectorFormat': 1, 'maxPowerAtSocket': 1, 'maximumCurrent': 1})


In [5]:
from collections import defaultdict

def find_paths(elem, target_name, current_path=None, results=None):
    if current_path is None:
        current_path = []
    if results is None:
        results = []

    name = localname(elem.tag)
    path = current_path + [name]

    if name == target_name:
        results.append("/".join(path))

    for child in list(elem):
        find_paths(child, target_name, path, results)

    return results

targets = [
    "nationalIdentifier", "name", "operator", "latitude", "longitude",
    "address", "addressLine", "postcode", "country",
    "typeOfSite", "serviceFacilityType",
    "connectorType", "connectorFormat", "chargingMode",
    "maxPowerAtSocket", "voltage", "maximumCurrent"
]

for t in targets:
    paths = find_paths(first_site, t)
    print(f"\nTARGET: {t}")
    for p in sorted(set(paths))[:20]:
        print(" ", p)


TARGET: nationalIdentifier

TARGET: name
  energyInfrastructureSite/energyInfrastructureStation/refillPoint/name
  energyInfrastructureSite/name
  energyInfrastructureSite/operator/name

TARGET: operator
  energyInfrastructureSite/operator

TARGET: latitude
  energyInfrastructureSite/locationReference/coordinatesForDisplay/latitude

TARGET: longitude
  energyInfrastructureSite/locationReference/coordinatesForDisplay/longitude

TARGET: address
  energyInfrastructureSite/locationReference/_locationReferenceExtension/facilityLocation/address

TARGET: addressLine
  energyInfrastructureSite/locationReference/_locationReferenceExtension/facilityLocation/address/addressLine

TARGET: postcode
  energyInfrastructureSite/locationReference/_locationReferenceExtension/facilityLocation/address/postcode

TARGET: country

TARGET: typeOfSite
  energyInfrastructureSite/typeOfSite

TARGET: serviceFacilityType

TARGET: connectorType
  energyInfrastructureSite/energyInfrastructureStation/refillPoint/conn

In [6]:
from collections import defaultdict

def find_paths(elem, target_name, current_path=None, results=None):
    if current_path is None:
        current_path = []
    if results is None:
        results = []

    name = localname(elem.tag)
    path = current_path + [name]

    if name == target_name:
        results.append("/".join(path))

    for child in list(elem):
        find_paths(child, target_name, path, results)

    return results

targets = [
    "nationalIdentifier", "name", "operator", "latitude", "longitude",
    "address", "addressLine", "postcode", "country",
    "typeOfSite", "serviceFacilityType",
    "connectorType", "connectorFormat", "chargingMode",
    "maxPowerAtSocket", "voltage", "maximumCurrent"
]

for t in targets:
    paths = find_paths(first_site, t)
    print(f"\nTARGET: {t}")
    for p in sorted(set(paths))[:20]:
        print(" ", p)


TARGET: nationalIdentifier

TARGET: name
  energyInfrastructureSite/energyInfrastructureStation/refillPoint/name
  energyInfrastructureSite/name
  energyInfrastructureSite/operator/name

TARGET: operator
  energyInfrastructureSite/operator

TARGET: latitude
  energyInfrastructureSite/locationReference/coordinatesForDisplay/latitude

TARGET: longitude
  energyInfrastructureSite/locationReference/coordinatesForDisplay/longitude

TARGET: address
  energyInfrastructureSite/locationReference/_locationReferenceExtension/facilityLocation/address

TARGET: addressLine
  energyInfrastructureSite/locationReference/_locationReferenceExtension/facilityLocation/address/addressLine

TARGET: postcode
  energyInfrastructureSite/locationReference/_locationReferenceExtension/facilityLocation/address/postcode

TARGET: country

TARGET: typeOfSite
  energyInfrastructureSite/typeOfSite

TARGET: serviceFacilityType

TARGET: connectorType
  energyInfrastructureSite/energyInfrastructureStation/refillPoint/conn

In [7]:
def print_attributes(elem, level=0, max_depth=4):
    if level > max_depth:
        return
    if elem.attrib:
        print("  " * level, localname(elem.tag), elem.attrib)
    for child in list(elem):
        print_attributes(child, level + 1, max_depth)

print_attributes(first_site, max_depth=5)

 energyInfrastructureSite {'id': 'OPMCKAGOAIX9NOFSBUXT', 'version': ''}
       value {'lang': 'es'}
   accessibility {'{http://www.w3.org/2001/XMLSchema-instance}nil': 'true'}
   operatingHours {'{http://www.w3.org/2001/XMLSchema-instance}type': 'fac:OperatingHoursSpecification', 'id': '24/7', 'version': ''}
   locationReference {'{http://www.w3.org/2001/XMLSchema-instance}type': 'loc:PointLocation'}
           addressLine {'order': '1'}
           addressLine {'order': '2'}
           addressLine {'order': '3'}
           addressLine {'order': '4'}
   operator {'{http://www.w3.org/2001/XMLSchema-instance}type': 'fac:OrganisationSpecification', 'id': 'ES*AEQ', 'version': ''}
         value {'lang': 'es'}
   energyInfrastructureStation {'id': 'OPMCKAGOAIX9NOFSBUXT_1', 'version': ''}
     refillPoint {'{http://www.w3.org/2001/XMLSchema-instance}type': 'egi:ElectricChargingPoint', 'id': 'E7U9O95AEBNUEAHON9IDLOHD6U0', 'version': ''}
           value {'lang': 'es'}
     refillPoint {'{http: